# Fine-Tuning Large Language Models with OpenAI API

This notebook sets up and sends a request to the OpenAI API to fine-tune a model. The process involves configuring the API endpoint, loading environment variables for authentication, and defining the data payload for the fine-tuning job. 

In [1]:
import requests
from openai import OpenAI
import os
from dotenv import load_dotenv
import json
import pandas as pd
from datetime import datetime


# Load OPENAI_API_KEY from .env file
load_dotenv()

# Initialize the OpenAI client
client = OpenAI()

In [30]:
def create_training_example(prompt_template: str, product1: str, product2: str, label: int):
    # Create the prompt with product descriptions
    prompt = insert_product_descriptions(prompt_template, product1, product2)
    if label == 1 or label == "1":
        label = "Yes"
    elif label == 0 or label == "0":
        label = "No"
    else:
        raise ValueError("Label is not 0 or 1")
    
    # Create the training example in the format required for fine-tuning
    return prompt, label

In [31]:
train_set = pd.read_pickle("../../data/wdc/train_small/preprocessed_wdcproducts80cc20rnd000un_train_small_with_explanation_all_fields_4_1.pkl.gz", compression="gzip")

from datasets import Dataset, load_dataset
# Create training examples
training_examples = []

for index, row in train_set.iterrows():
    product_1 = serialize_product(row, "left")
    product_2 = serialize_product(row, "right")
    label = str(row.get("label"))  # Convert label to string
    
    prompt, response = create_training_example(PROMPT_TEMPLATE, product_1, product_2, label)
    training_examples.append({"prompt": prompt, "completion": response})

# Convert to Dataset format
dataset = Dataset.from_list(training_examples)

In [32]:
pd.DataFrame(training_examples)["completion"].value_counts()


completion
Npo    2000
Yes     500
Name: count, dtype: int64

In [20]:
dataset_csv = load_dataset('csv', data_files="../../data/wdc/train_small/preprocessed_wdcproducts80cc20rnd000un_train_small_brand_title_price_currency.csv", split="train")

Generating train split: 0 examples [00:00, ? examples/s]

/ceph/aasteine/fine-tuning-paper/venv/lib64/python3.9/site-packages/datasets/download/streaming_download_manager.py:778: FutureWarning: The 'verbose' keyword in pd.read_csv is deprecated and will be removed in a future version.
  return pd.read_csv(xopen(filepath_or_buffer, "rb", download_config=download_config), **kwargs)


In [21]:
dataset_csv

Dataset({
    features: ['prompt', 'completion'],
    num_rows: 2500
})

In [22]:
dataset_csv[0]

{'prompt': "Do the two product descriptions refer to the same real-world product? Entity 1: 'nan HDD 35 4TB Seagate IronWolf Pro NAS ST4000NE001 154.10 nan'. Entity 2: 'nan HD 3,5 4TB 7200RPM IRONWOLF PRO 128 MB SATA3 SEAGATE  153.99 EUR'.",
 'completion': 'Yes'}

In [18]:
dataset

Dataset({
    features: ['prompt', 'completion'],
    num_rows: 2500
})

In [23]:
dataset[0]

{'prompt': 'Do the two product descriptions refer to the same real-world product? Entity 1: [TITLE] HDD 35 4TB Seagate IronWolf Pro NAS ST4000NE001 [PRICE] 154.10. Entity 2: [TITLE] HD 3,5 4TB 7200RPM IRONWOLF PRO 128 MB SATA3 SEAGATE [PRICE] 153.99 [CURRENCY] EUR.',
 'completion': 'Yes'}

## Transform data from local format to jsonl

In [9]:
PROMPT_TEMPLATE = "Do the two product descriptions refer to the same real-world product? Entity 1: 'Entity 1'. Entity 2: 'Entity 2'."
MODEL_NAME = "gpt-4.1-mini-2025-04-14"
VALIDATION_FILE_ID = "file-4ncMP2qadRFkD9KiLPPQgz"

In [3]:
def serialize_product(row, side='left'):
    """Serialize product attributes from a row, only including non-NaN values.
    
    Args:
        row: DataFrame row containing product information
        side: 'left' or 'right' to indicate which product to serialize
        
    Returns:
        str: Serialized product string with non-NaN attributes
    """
    attributes = []
    
    # Add brand if available
    brand = row[f'brand_{side}']
    if pd.notna(brand):
        attributes.append(f"[BRAND] {brand}")
    
    # Add title (required)
    title = row[f'title_{side}']
    if pd.notna(title):
        attributes.append(f"[TITLE] {title}")
    
    # Add description if available
    description = row[f'description_{side}']
    if pd.notna(description):
        attributes.append(f"[DESCRIPTION] {description}")
    
    # Add price and currency if available
    price = row[f'price_{side}']
    if pd.notna(price):
        price_str = f"[PRICE] {price}"
        currency = row[f'priceCurrency_{side}']
        if pd.notna(currency):
            price_str += f" [CURRENCY] {currency}"
        attributes.append(price_str)
    
    return " ".join(attributes)


In [4]:
def insert_product_descriptions(prompt_template: str, product1: str, product2: str):
    # Replace placeholder texts with actual product descriptions
    prompt = prompt_template.replace("'Entity 1'", product1).replace("'Entity 2'", product2)
    return prompt

def create_training_example(prompt_template: str, product1: str, product2: str, label: int):
    # Create the prompt with product descriptions
    prompt = insert_product_descriptions(prompt_template, product1, product2)
    if label == 1 or label == "1":
        label = "Yes"
    elif label == 0 or label == "0":
        label = "No"
    else:
        ValueError("Label is not 0 or 1")
    
    # Create the training example in the format required for fine-tuning
    return {
        "messages": [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": label}
        ]
    }


In [6]:
def train_based_on_csv(train_set_path: str, run_name: str, model_name: str, validation_file_id: str):
    # Load the test set
    train_set = pd.read_csv(train_set_path)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_dir = f"../../results/{model_name}/all-attributes/{run_name}/{timestamp}"
    os.makedirs(output_dir, exist_ok=True)

    # Create training examples
    training_examples = []
    print(f"Creating training examples for {run_name}. The dataset has {len(train_set)} pairs each.")

    
    for index, row in train_set.iterrows():
        prompt = row["prompt"]
        label = row["completion"]
        
        example = {
            "messages": [
                {"role": "user", "content": prompt},
                {"role": "assistant", "content": label}
            ]
        }
        
        training_examples.append(example)

    # Save the training file
    training_file_path = os.path.join(output_dir, "training.jsonl")
    with open(training_file_path, "w") as f:
        for example in training_examples:
            f.write(json.dumps(example) + "\n")
        
    # Upload the training file using the SDK
    training_file = client.files.create(
        file=open(training_file_path, "rb"),
        purpose="fine-tune"
    )

    print(f"Training file uploaded successfully. File ID: {training_file.id}")

    # Start the fine-tuning job using the SDK
    fine_tune_job = client.fine_tuning.jobs.create(
        training_file=training_file.id,
        validation_file=validation_file_id,
        model=model_name,
        hyperparameters={
            "n_epochs": 5
        },
        seed=42
    )

    print(f"Fine-tuning job started successfully. Job ID: {fine_tune_job.id}")

    # Save the run configuration
    run_config = {
        "run_name": run_name,
        "timestamp": timestamp,
        "model": MODEL_NAME,
        "file_id": training_file.id,
        "job_id": fine_tune_job.id,
        "status": fine_tune_job.status,
        "created_at": fine_tune_job.created_at
    }

    with open(os.path.join(output_dir, "fine-tune-run_config.json"), "w") as f:
        json.dump(run_config, f, indent=2)

    print(f"Run configuration and training files saved to: {output_dir}")

In [7]:
def train_based_on_pickle(train_set_path: str, run_name: str, model_name: str, validation_file_id: str, label_col_name:str="label"):
    # Load the test set
    train_set = pd.read_pickle(train_set_path)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_dir = f"../../results/{model_name}/all-attributes/{run_name}/{timestamp}"
    os.makedirs(output_dir, exist_ok=True)

    # Create training examples
    training_examples = []
    print(f"Creating training examples for {run_name}. The dataset has {len(train_set)} pairs each.")

        
    for index, row in train_set.iterrows():
        product_1 = serialize_product(row, "left")
        product_2 = serialize_product(row, "right")
        label = str(row.get(label_col_name))  # Convert label to string
        
        example = create_training_example(PROMPT_TEMPLATE, product_1, product_2, label)
        training_examples.append(example)

    # Save the training file
    training_file_path = os.path.join(output_dir, f"training-{run_name}.jsonl")
    with open(training_file_path, "w") as f:
        for example in training_examples:
            f.write(json.dumps(example) + "\n")


    # Upload the training file using the SDK
    training_file = client.files.create(
        file=open(training_file_path, "rb"),
        purpose="fine-tune"
    )

    print(f"Training file uploaded successfully. File ID: {training_file.id}")

    # Start the fine-tuning job using the SDK
    fine_tune_job = client.fine_tuning.jobs.create(
        training_file=training_file.id,
        validation_file=validation_file_id,
        model=model_name,
        hyperparameters={
            "n_epochs": 5
        },
        seed=42
    )

    print(f"Fine-tuning job started successfully. Job ID: {fine_tune_job.id}")

    # Save the run configuration
    run_config = {
        "run_name": run_name,
        "timestamp": timestamp,
        "model": MODEL_NAME,
        "file_id": training_file.id,
        "job_id": fine_tune_job.id,
        "status": fine_tune_job.status,
        "created_at": fine_tune_job.created_at
    }

    with open(os.path.join(output_dir, "fine-tune-run_config.json"), "w") as f:
        json.dump(run_config, f, indent=2)

    print(f"Run configuration and training files saved to: {output_dir}")

## Upload validation file

In [12]:
validation_set_path = "../../data/wdc/validation/preprocessed_wdcproducts80cc20rnd000un_valid_small.pkl.gz"
# Load the test set
validation_set = pd.read_pickle(validation_set_path)

# Create training examples
validation_examples = []
print(f"The validation set has {len(validation_set)} pairs each.")

for index, row in validation_set.iterrows():
    product_1 = serialize_product(row, "left")
    product_2 = serialize_product(row, "right")
    label = str(row.get('label'))  # Convert label to string
    pair_id = row['pair_id']
    
    example = create_training_example(PROMPT_TEMPLATE, product_1, product_2, label)
    validation_examples.append(example)

# Save the training file
with open(validation_set_path.replace(".pkl.gz", "_all_attributes.jsonl"), "w") as f:
    for example in validation_examples:
        f.write(json.dumps(example) + "\n")


# Upload the training file using the SDK
validation_file = client.files.create(
    file=open(validation_set_path.replace(".pkl.gz", ".jsonl"), "rb"),
    purpose="fine-tune"
)

print(f"Validation file uploaded successfully. File ID: {validation_file.id}")


The validation set has 2500 pairs each.
Validation file uploaded successfully. File ID: file-4ncMP2qadRFkD9KiLPPQgz


## Small wdc no augmentations

In [14]:
train_based_on_pickle("../../data/wdc/train_small/preprocessed_wdcproducts80cc20rnd000un_train_small.pkl.gz", "wdc_all_attributes_no_augmentation", MODEL_NAME, VALIDATION_FILE_ID)

Creating training examples for wdc_all_attributes_no_augmentation. The dataset has 2500 pairs each.
Training file uploaded successfully. File ID: file-Wqu6U5EF5zqPt9a7aSbZMF
Fine-tuning job started successfully. Job ID: ftjob-utnX87bX9a4DsQeOojsG2NNb
Run configuration and training files saved to: ../../results/gpt-4.1-mini-2025-04-14/wdc_all_attributes_no_augmentation/20250509_150226


## Train with explanation

In [17]:
train_based_on_pickle("../../data/wdc/train_small/preprocessed_wdcproducts80cc20rnd000un_train_small_with_explanation_all_fields_4_1.pkl.gz", "wdc_all_attributes_explanation", MODEL_NAME, VALIDATION_FILE_ID, label_col_name="explanation")

Creating training examples for wdc_all_attributes_explanation. The dataset has 2500 pairs each.
Training file uploaded successfully. File ID: file-L9twzZ4nu99JQC4BTW9MKU
Fine-tuning job started successfully. Job ID: ftjob-VYhJoBFVNYq0bqvUjcFgkCpa
Run configuration and training files saved to: ../../results/gpt-4.1-mini-2025-04-14/wdc_all_attributes_explanation/20250509_150722


## Train simple swapping

In [8]:
train_based_on_pickle("../../data/wdc/train_small/augmentation/preprocessed_wdcproducts80cc20rnd000un_train_small_explanations_41_swapped_matching_examples.pkl.gz", "fine-tune-wdc-simple-swapping-all-attributes", MODEL_NAME, VALIDATION_FILE_ID)

Creating training examples for fine-tune-wdc-simple-swapping-all-attributes. The dataset has 5455 pairs each.
Training file uploaded successfully. File ID: file-3qUobm4H3i9RQCkwCQoqcf
Fine-tuning job started successfully. Job ID: ftjob-AA26QI1hGWEGzQkhiaJ3QDvE
Run configuration and training files saved to: ../../results/gpt-4.1-mini-2025-04-14/all-attributes/fine-tune-wdc-simple-swapping-all-attributes/20250510_112703


## Train based on 10% swapping

In [9]:
train_based_on_pickle("../../data/wdc/train_small/augmentation/preprocessed_wdcproducts80cc20rnd000un_train_small_explanations_41_swapped_matching_examples_0_10_permutations.pkl.gz", "fine-tune-wdc-simple-swapping-10-all-attributes", MODEL_NAME, VALIDATION_FILE_ID)

Creating training examples for fine-tune-wdc-simple-swapping-10-all-attributes. The dataset has 13351 pairs each.
Training file uploaded successfully. File ID: file-FGJ2MUevssYgJj3V8XjxuT
Fine-tuning job started successfully. Job ID: ftjob-XvwfA8HuxRAp8FpaEWQ7jmYF
Run configuration and training files saved to: ../../results/gpt-4.1-mini-2025-04-14/all-attributes/fine-tune-wdc-simple-swapping-10-all-attributes/20250510_113030
